In [1]:
# Install packages
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

In [3]:
data = pd.read_csv("../data/Data_set_new.csv")

data.head()

,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-8.839529e+18,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5.365332e+18,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,NaN,NaN,NaN,NaN,2020-08-24,3.220506,3.501733,7.547777,14.560315,270,1
3,7.750424e+18,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
4,NaN,NaN,NaN,NaN,2021-03-21,4.218409,5.098378,9.198853,13.970645,198,2


In [4]:

# Convert EventDate to datetime
data["EventDate"] = pd.to_datetime(data["EventDate"])

# Check the data type
print(data["EventDate"].dtype)

datetime64[us]


In [5]:
# Standardize column names
data.columns = [
    "animal_id",
    "lactation_number",
    "days_in_milk",
    "reproduction_status",
    "event_date",
    "avg_milk_flow",
    "flow_30_60_session",
    "yield_first_2min_session",
    "yield_session",
    "duration_session_sec",
    "milking"
]

# Check column names
print(data.columns.tolist())

['animal_id', 'lactation_number', 'days_in_milk', 'reproduction_status', 'event_date', 'avg_milk_flow', 'flow_30_60_session', 'yield_first_2min_session', 'yield_session', 'duration_session_sec', 'milking']


In [6]:
# Check missing values and proportions
missing_count = data.isnull().sum()
missing_percent = data.isnull().mean() * 100

missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percent": missing_percent
})

print(missing_summary)

                          Missing_Count  Missing_Percent
animal_id                       1701003        20.022586
lactation_number                1701003        20.022586
days_in_milk                    1701007        20.022633
reproduction_status             1701003        20.022586
event_date                            0         0.000000
avg_milk_flow                       172         0.002025
flow_30_60_session                    0         0.000000
yield_first_2min_session              0         0.000000
yield_session                         0         0.000000
duration_session_sec                  0         0.000000
milking                               0         0.000000


Remove Duplicated

In [10]:
# Check duplicates before removal
print("Duplicate rows before cleaning:", data.duplicated().sum())

Duplicate rows before cleaning: 60653


In [11]:
# Show duplicate rows
duplicates = data[data.duplicated(keep=False)]

duplicates.head(20)

,animal_id,lactation_number,days_in_milk,reproduction_status,event_date,avg_milk_flow,flow_30_60_session,yield_first_2min_session,yield_session,duration_session_sec,milking
95,3.586802e+18,2.0,156.0,Pregnant,2020-05-27,4.399846,5.801446,9.974496,11.113013,150,2
208,5.438336e+18,1.0,78.0,Bred,2019-12-18,2.721554,2.599084,5.098378,11.611965,256,2
304,-8.755941e+18,2.0,280.0,Bred,2019-12-19,2.222603,2.000342,4.749112,11.022295,289,2
394,2.814185e+18,1.0,210.0,Open,2021-04-20,3.810176,4.599427,8.386923,11.294450,177,2
409,8.149891e+18,1.0,101.0,Bred,2021-10-07,3.492661,3.197826,6.173392,16.873636,288,1
413,3.455214e+18,2.0,268.0,Pregnant,2021-04-20,3.900894,4.799007,8.949377,13.335616,202,1
466,-6.137090e+18,2.0,259.0,Pregnant,2020-05-27,3.401943,4.191193,7.520561,16.011811,275,2
563,4.962509e+18,2.0,104.0,Bred,2021-05-16,3.900894,4.998588,8.550216,20.003424,307,3
631,6.117359e+18,3.0,53.0,Open,2021-06-08,4.490564,2.599084,8.926698,21.817793,287,3
648,-5.942447e+18,1.0,256.0,Pregnant,2019-12-19,3.583380,4.499636,8.096624,9.979032,166,2


In [17]:
# Remove exact duplicate rows
data = data.drop_duplicates().copy()

# the first occurrence of a repeated row is kept
# later identical rows are counted as duplicates

In [18]:
# Check again
print("Duplicate rows after cleaning:", data.duplicated().sum())
print("Rows after cleaning:", len(data))

Duplicate rows after cleaning: 0
Rows after cleaning: 8434768


Remove Null

In [20]:
# Check number of null values in each column
data.isnull().sum()

animal_id                   1696870
lactation_number            1696870
days_in_milk                1696874
reproduction_status         1696870
event_date                        0
avg_milk_flow                   172
flow_30_60_session                0
yield_first_2min_session          0
yield_session                     0
duration_session_sec              0
milking                           0
dtype: int64

In [21]:
# Drop rows with missing avg_milk_flow
data = data.dropna(subset=["avg_milk_flow"])

# Check the result
print("Remaining rows:", len(data))
print("Missing avg_milk_flow:", data["avg_milk_flow"].isnull().sum())

Remaining rows: 8434596
Missing avg_milk_flow: 0


Remove Zero

In [24]:
# Remove rows where avg_milk_flow is 0
data = data[data["avg_milk_flow"] != 0].copy()

# Check the result
print("Remaining rows:", len(data))

Remaining rows: 8434392


Remove Outliers

In [26]:
# Check ranges of numerical variables
variables_to_check = [
    "lactation_number",
    "days_in_milk",
    "avg_milk_flow",
    "flow_30_60_session",
    "yield_first_2min_session",
    "yield_session",
    "duration_session_sec",
    "milking"
]

data[variables_to_check].describe()

,lactation_number,days_in_milk,avg_milk_flow,flow_30_60_session,yield_first_2min_session,yield_session,duration_session_sec,milking
count,6.737603e+06,6.737599e+06,8.434392e+06,8.434392e+06,8.434392e+06,8.434392e+06,8.434392e+06,8.434392e+06
mean,2.293939e+00,1.629379e+02,3.526017e+00,3.751318e+00,7.491111e+00,1.392733e+01,2.368538e+02,1.999986e+00
std,1.387634e+00,1.055156e+02,7.290172e-01,1.485699e+00,2.014106e+00,3.643992e+00,5.483618e+01,8.165099e-01
min,1.000000e+00,1.000000e+00,4.082331e-01,0.000000e+00,2.000342e+00,5.034875e+00,1.000000e+02,1.000000e+00
25%,1.000000e+00,7.600000e+01,2.993710e+00,2.798665e+00,6.023707e+00,1.143053e+01,1.970000e+02,1.000000e+00
50%,2.000000e+00,1.540000e+02,3.492661e+00,3.900894e+00,7.525097e+00,1.374385e+01,2.310000e+02,2.000000e+00
75%,3.000000e+00,2.370000e+02,3.991613e+00,4.898798e+00,8.999273e+00,1.632933e+01,2.720000e+02,3.000000e+00
max,1.200000e+01,8.700000e+02,2.349608e+01,1.050066e+01,1.199752e+01,5.483932e+01,7.850000e+02,3.000000e+00


In [28]:
# Check potential outliers using Z-score

continuous_vars = [
    "days_in_milk",
    "avg_milk_flow",
    "flow_30_60_session",
    "yield_first_2min_session",
    "yield_session",
    "duration_session_sec"
]

for var in continuous_vars:
    z = (data[var] - data[var].mean()) / data[var].std()
    outliers = (abs(z) > 3).sum()

    print(var, ":", outliers)

days_in_milk : 44121
avg_milk_flow : 3599
flow_30_60_session : 839
yield_first_2min_session : 0
yield_session : 20622
duration_session_sec : 31948


In [29]:
# Check actual values identified as potential outliers

for var in continuous_vars:
    z = (data[var] - data[var].mean()) / data[var].std()
    outlier_values = data.loc[abs(z) > 3, var]

    print(
        var,
        "| min:", outlier_values.min(),
        "| max:", outlier_values.max()
    )

days_in_milk | min: 480.0 | max: 870.0
avg_milk_flow | min: 0.408233133 | max: 23.496084766
flow_30_60_session | min: 8.300740371000002 | max: 10.5006633655
yield_first_2min_session | min: nan | max: nan
yield_session | min: 24.902221113 | max: 54.839317533000006
duration_session_sec | min: 402 | max: 785


In [36]:
# Remove all rows containing potential outliers (|Z| > 3)

outlier_mask = False

for var in continuous_vars:
    z = (data[var] - data[var].mean()) / data[var].std()
    outlier_mask = outlier_mask | (abs(z) > 3)

data = data[~outlier_mask].copy()

print("Remaining rows:", len(data))

Remaining rows: 8336598
